# Derivative pricing using Neural Networks: Non-Parametric Data Preparation

In [ ]:
from DerivativeUtil import download_and_extract_zip, load_stockdata_from_directory, load_optiondata_from_directory
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime
from pandas.tseries.offsets import BDay
import yfinance as yf
from DerivModel import black_scholes

## Download and prepare data
Option Data provide free option price data from January to June 2013. This notebook downloads and processes data from this period relating to S&P 500 European Options. 

In [ ]:
url_base = 'https://optiondata.org/2013-0'
data_path = './data'

In [ ]:
for i in range(1, 7):
    full_url = url_base + str(i) + '.zip'
    download_and_extract_zip(full_url, data_path)

In [ ]:
dfoptions = load_optiondata_from_directory(data_path)

In [ ]:
dfoptions.head()

Only the data for S&P 500 is used so the rest is dropped

In [ ]:
dfspxop = dfoptions[dfoptions['underlying'] == 'SPX']

In [ ]:
len(dfspxop)

We need to convert dates to Pandas datetimes and to calculate the maturity in years.

In [ ]:
dfspxop['quote_date'] = pd.to_datetime(dfspxop['quote_date'])
dfspxop['expiration'] = pd.to_datetime(dfspxop['expiration'])
dfspxop['maturity'] = (dfspxop['expiration'] - dfspxop['quote_date']).dt.days / 365.25

No interest rate data is provided with the option price data and so this needs to be sources elsewhere. Historical zero rates can be obtained from [Federal Reserve Economic Data (FRED)](https://fred.stlouisfed.org/). This can be downloaded using a python package if desired, but here the website was used to directly download csv files. Four different historical time series are needed for 1,2,3, and 5 year zero rates:
- https://fred.stlouisfed.org/series/THREEFY1
- https://fred.stlouisfed.org/series/THREEFY2
- https://fred.stlouisfed.org/series/THREEFY3
- https://fred.stlouisfed.org/series/THREEFY5

The csv data is loaded and merged into a single dataframe. Note that the data is provided in percentage terms and hence needs to be divided by 100 to express them as rates.

In [ ]:
dfzero1yr = pd.read_csv("THREEFY1.csv").set_index("DATE")
dfzero2yr = pd.read_csv("THREEFY2.csv").set_index("DATE")
dfzero3yr = pd.read_csv("THREEFY3.csv").set_index("DATE")
dfzero5yr = pd.read_csv("THREEFY5.csv").set_index("DATE")

In [ ]:
merged_df = pd.concat([dfzero1yr, dfzero2yr, dfzero3yr, dfzero5yr], axis=1)

In [ ]:
merged_df.rename(columns={"THREEFY1": 1.0}, inplace=True)
merged_df.rename(columns={"THREEFY2": 2.0}, inplace=True)
merged_df.rename(columns={"THREEFY3": 3.0}, inplace=True)
merged_df.rename(columns={"THREEFY5": 5.0}, inplace=True)

In [ ]:
for col in merged_df.columns:
    merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')

In [ ]:
merged_df.index = pd.to_datetime(merged_df.index)

In [ ]:
merged_df = merged_df / 100.0

An interpolation function is defined and then used with a lambda to calculate the rate to maturity for each option in the dataframe.

In [ ]:
def interpolate_zero_rate(df, observation_date, target_maturity):
    rates = df.loc[observation_date]
    y = rates.to_numpy()
    x = rates.index.to_numpy()
    return np.interp(target_maturity, x, y)

In [ ]:
dfspxop['rate'] = dfspxop.apply(lambda row: interpolate_zero_rate(merged_df, row['quote_date'], row['maturity']), axis=1)

In [ ]:
dfspxop

The S&P 500 closing prices are supplied with the option data but given historical volatility is used in the examples more data is needed. This is sought from Yahoo Finance using the yfinance Python package. A time series starting 90 business days before the start of the option data is used. This allows the rolling historical volatility for 60, 40 and 20 days to be calculated. This is done and appended to the dataframe.

In [ ]:
def fetch_sp500_closing_prices(start_date, end_date):
    # Convert start_date and end_date to datetime objects if they are strings
    if isinstance(start_date, str):
        start_date = datetime.datetime.strptime(start_date, '%Y-%m-%d')
    if isinstance(end_date, str):
        end_date = datetime.datetime.strptime(end_date, '%Y-%m-%d')

    # Calculate start_date - 90 business days
    adjusted_start_date = start_date - BDay(90)

    # Fetch data
    df = yf.download('^GSPC', start=adjusted_start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))

    return df

In [ ]:
# Example usage
start_date = '2013-01-01'
end_date = '2013-07-01'
closing_prices = fetch_sp500_closing_prices(start_date, end_date)

In [ ]:
# Calculate daily logarithmic returns
closing_prices['Log_Returns'] = np.log(closing_prices['Close'] / closing_prices['Close'].shift(1))

# Calculate rolling standard deviation
closing_prices['60D_HV'] = closing_prices['Log_Returns'].rolling(window=60).std() * np.sqrt(252)  # 252 trading days in a year
closing_prices['40D_HV'] = closing_prices['Log_Returns'].rolling(window=40).std() * np.sqrt(252)
closing_prices['20D_HV'] = closing_prices['Log_Returns'].rolling(window=20).std() * np.sqrt(252)

# Drop the 'Log_Returns' column if you don't want to keep it in the final DataFrame
closing_prices = closing_prices.drop(columns=['Log_Returns'])

In [ ]:
closing_prices.tail()

In [ ]:
print(closing_prices.columns)

In [ ]:
hv_cols = closing_prices[['Close', '20D_HV', '40D_HV', '60D_HV']].copy()
hv_cols.columns = hv_cols.columns.get_level_values(0)
dfspxop = pd.merge(dfspxop, hv_cols, left_on='quote_date', right_index=True, how='left')

In [ ]:
dfspxop.columns

The closing prices and the historical volatility values are now merged to the option dataframe.

In [ ]:
close_series = closing_prices[('Close', '^GSPC')]
dfspxop['underlying'] = dfspxop['quote_date'].map(close_series)

In [ ]:
dfspxop.head()

Bid and Ask size and NaNs so these columns are dropped

In [ ]:
dfspxop = dfspxop.drop(columns=['bid_size', 'ask_size'])

If both the bid and the ask prices are zero then it is an indication that row of data is invalid so these rows are dropped.

In [ ]:
dfspxop = dfspxop[~((dfspxop['bid'] == 0.0) & (dfspxop['ask'] == 0.0))]

In [ ]:
option_type = 'call'
strike = 100
vol = 0.2
T = 1.0
spot = 105
rate = 0.05
bsvalue = black_scholes(option_type, strike, vol, T, spot, rate)

In [ ]:
bsvalue.value()

For each remaining row the Black-Scholes is calculated using the supplied Implied Volatility.

In [ ]:
bs_lst = list()
for _, row in dfspxop.iterrows():
    bsres = black_scholes(row['type'], row['strike'], row['implied_volatility'],
                          row['maturity'], row['underlying'], row['rate'])
    bs_lst.append(bsres.value())

In [ ]:
dfspxop['BlackScholes'] = bs_lst

The option pricing data only contains bid and ask prices and so these are averaged to give a mid price.

In [ ]:
dfspxop['mid'] = (dfspxop['bid'] + dfspxop['ask']) / 2

Rows with mid prices too far from the Black-Scholes value are also dropped as these points are invalid.

In [ ]:
dfspxop['error'] = (dfspxop['BlackScholes'] - dfspxop['mid'])

In [ ]:
dfspxop = dfspxop[dfspxop['error'] < 100.0]

In [ ]:
out_file = 'SPXOptions.csv'
dfspxop.to_csv(out_file, index=False)